In [ ]:
# 0️⃣ Imports and setup
import os
import json
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_recall_curve, auc, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split, ParameterGrid
import joblib

In [ ]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [ ]:
BASE_DIR = '/content/drive/MyDrive/Anomaly_Det/outputs/phase2'
PHASE2_OUT = os.path.join(BASE_DIR, 'phase2')
os.makedirs(os.path.join(PHASE2_OUT, 'models'), exist_ok=True)
os.makedirs(os.path.join(PHASE2_OUT, 'metrics'), exist_ok=True)
print("Directories ready")

Directories ready


In [ ]:

# 1️⃣ Load dataset / features
features_path = "/content/drive/MyDrive/Anomaly_Det/outputs/phase1/data/features_v1.parquet"
features = pd.read_parquet(features_path)
features.columns = features.columns.str.strip()
id_cols = [c for c in ['CLM_ID', 'DESYNPUF_ID'] if c in features.columns]
X = features.drop(columns=id_cols) if id_cols else features.copy()

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Anomaly_Det/outputs/phase1/data/features_v1.parquet'

In [ ]:
# Load feature table v1 (created in Phase 1)
features_path = "/content/drive/MyDrive/Anomaly_Det/outputs/phase1/data/features_v1.parquet"
features = pd.read_parquet(features_path)
print('Features shape:', features.shape)
# Drop ID columns if present and keep only numeric model features
id_cols = [c for c in ['CLM_ID', 'DESYNPUF_ID'] if c in features.columns]
X = features.drop(columns=id_cols) if id_cols else features.copy()

# Work with numeric columns used by models (isolation forest uses numeric)
X_num = X.select_dtypes(include=[np.number]).copy()
print('Numeric feature shape:', X_num.shape)

Features shape: (174418, 16)
Numeric feature shape: (174418, 14)


In [ ]:
X_train_full, X_val_clean = train_test_split(X_num, test_size=0.2, random_state=RANDOM_SEED)
print('Train rows:', len(X_train_full), 'Val rows (clean):', len(X_val_clean))

Train rows: 139534 Val rows (clean): 34884


In [ ]:
# Re-implement injection function (same logic as Phase 2) to create a labeled validation set
import numpy as np

def inject_anomalies(X, injection_rate=0.03, seed=RANDOM_SEED):
    X = X.copy().reset_index(drop=True)
    n = len(X)
    n_anomalies = int(n * injection_rate)
    np.random.seed(seed)
    positions = np.random.choice(n, n_anomalies, replace=False)

    # Prepare y labels
    y = np.zeros(n, dtype=int)

    # Type distribution (same ratios used earlier)
    n_monetary = int(n_anomalies * 0.30)
    n_temporal = int(n_anomalies * 0.20)
    n_coding = int(n_anomalies * 0.30)
    n_duplicate = n_anomalies - n_monetary - n_temporal - n_coding

    monetary_pos = positions[:n_monetary]
    temporal_pos = positions[n_monetary:n_monetary+n_temporal]
    coding_pos = positions[n_monetary+n_temporal:n_monetary+n_temporal+n_coding]
    duplicate_pos = positions[n_monetary+n_temporal+n_coding:]

    # Monetary outliers: inflate payment features if present
    for pos in monetary_pos:
        if 'log_payment' in X.columns:
            X.loc[pos, 'log_payment'] = X.loc[pos, 'log_payment'] * np.random.uniform(3, 5)
        if 'payment_zscore' in X.columns:
            X.loc[pos, 'payment_zscore'] = X.loc[pos, 'payment_zscore'] * np.random.uniform(3, 5)

    # Temporal impossibilities: set claim_duration extreme values
    for pos in temporal_pos:
        if 'claim_duration' in X.columns:
            X.loc[pos, 'claim_duration'] = int(np.random.choice([400, 500, -10]))

    # Coding anomalies: set total_code_count to 0 or very high
    for pos in coding_pos:
        if 'total_code_count' in X.columns:
            X.loc[pos, 'total_code_count'] = int(np.random.choice([0, 50]))

    # Duplicates with noise: small multiplicative noise
    for pos in duplicate_pos:
        noise = np.random.uniform(0.95, 1.05)
        # apply noise to all numeric columns that look monetary-like
        if 'log_payment' in X.columns:
            X.loc[pos, 'log_payment'] = X.loc[pos, 'log_payment'] * noise

    y[positions] = 1
    return X, y

# Create injected validation set
X_val_injected, y_val = inject_anomalies(X_val_clean, injection_rate=0.03, seed=RANDOM_SEED)
print('Injected anomalies in validation:', int(y_val.sum()))

Injected anomalies in validation: 1046


In [ ]:
# Scale numeric features for LOF/OCSVM compatibility; IsolationForest can use unscaled but we'll keep scaling consistent
scaler = StandardScaler()
X_train_num = X_train_full.fillna(0)
X_train_scaled = scaler.fit_transform(X_train_num)
X_val_scaled = scaler.transform(X_val_injected.fillna(0))

# For IsolationForest we'll use the unscaled X_train_num and X_val_injected (both filled)
X_train_if = X_train_num.fillna(0)
X_val_if = X_val_injected.fillna(0)

In [ ]:
# Manual grid search (evaluate on injected validation using PR-AUC)
from sklearn.metrics import precision_recall_curve, auc
from sklearn.model_selection import ParameterGrid

param_grid = {
    'n_estimators': [100, 200, 400],
    'max_samples': ['auto', 0.5, 0.8],
    'contamination': [0.01, 0.03, 0.05],
    'max_features': [1.0, 0.8],
    'bootstrap': [False, True]
}

grid = list(ParameterGrid(param_grid))
print('Grid size:', len(grid))

results = []
for i, params in enumerate(grid):
    # Train IsolationForest on clean train set
    clf = IsolationForest(random_state=RANDOM_SEED, **params)
    try:
        clf.fit(X_train_if)
        # Use negative score_samples so higher = more anomalous
        scores = -clf.score_samples(X_val_if)
        # Compute PR-AUC
        precision, recall, _ = precision_recall_curve(y_val, scores)
        pr_auc = auc(recall, precision)
        results.append({**params, 'pr_auc': pr_auc})
    except Exception as e:
        results.append({**params, 'pr_auc': float('nan'), 'error': str(e)})
    if (i+1) % 10 == 0:
        print(f'Completed {i+1}/{len(grid)}')

results_df = pd.DataFrame(results).sort_values('pr_auc', ascending=False).reset_index(drop=True)
results_df.to_csv(os.path.join(PHASE2_OUT, 'metrics', 'if_grid_results.csv'), index=False)
print('Top results:')
print(results_df.head(10))

Grid size: 108
Completed 10/108


KeyboardInterrupt: 

In [ ]:
# Select best params and refit on full train (train + validation clean if desired)
best_row = results_df.dropna(subset=['pr_auc']).iloc[0]
best_params = {k: best_row[k] for k in param_grid.keys()}
print('Best params:', best_params)

# Refit on the full training data (optionally include X_val_clean)
X_refit = pd.concat([X_train_if, X_val_if[y_val==0]]).reset_index(drop=True)
clf_best = IsolationForest(random_state=RANDOM_SEED, **best_params)
clf_best.fit(X_refit)

# Save model and metadata
model_path = os.path.join(PHASE2_OUT, 'models', f'isolationforest_best_{datetime.now().strftime("%Y%m%d_%H%M%S")}.pkl')
joblib.dump(clf_best, model_path)
print('Saved best model to', model_path)

# Save summary
summary = {
    'best_params': best_params,
    'best_pr_auc': float(best_row['pr_auc'])
}
with open(os.path.join(PHASE2_OUT, 'models', 'if_grid_search_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print('Summary saved')

In [ ]:
# Quick visualization of top 10 grid results
plt.figure(figsize=(10, 4))
plot_df = results_df.head(20).copy()
plt.barh(range(len(plot_df)), plot_df['pr_auc'][::-1])
plt.yticks(range(len(plot_df)), (plot_df.index[::-1].astype(str)))
plt.xlabel('PR-AUC')
plt.title('Top grid candidates (by PR-AUC)')
plt.tight_layout()
plt.savefig(os.path.join(PHASE2_OUT, 'metrics', 'if_grid_top20.png'), dpi=300)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Scatter 1: contamination vs pr_auc (colored by n_estimators)
scatter1 = axes[0, 0].scatter(results_df['contamination'], results_df['pr_auc'],
                               c=results_df['n_estimators'], cmap='viridis', s=80, alpha=0.6)
axes[0, 0].set_xlabel('Contamination')
axes[0, 0].set_ylabel('PR-AUC')
axes[0, 0].set_title('Contamination vs PR-AUC (colored by n_estimators)')
plt.colorbar(scatter1, ax=axes[0, 0], label='n_estimators')

# Scatter 2: max_samples vs pr_auc
max_samples_numeric = results_df['max_samples'].apply(lambda x: x if isinstance(x, (int, float)) else 1.0)
scatter2 = axes[0, 1].scatter(max_samples_numeric, results_df['pr_auc'],
                               c=results_df['contamination'], cmap='plasma', s=80, alpha=0.6)
axes[0, 1].set_xlabel('Max Samples')
axes[0, 1].set_ylabel('PR-AUC')
axes[0, 1].set_title('Max Samples vs PR-AUC (colored by contamination)')
plt.colorbar(scatter2, ax=axes[0, 1], label='contamination')

# Scatter 3: n_estimators vs pr_auc (colored by bootstrap)
bootstrap_colors = results_df['bootstrap'].map({True: 1, False: 0})
scatter3 = axes[1, 0].scatter(results_df['n_estimators'], results_df['pr_auc'],
                               c=bootstrap_colors, cmap='coolwarm', s=80, alpha=0.6)
axes[1, 0].set_xlabel('N Estimators')
axes[1, 0].set_ylabel('PR-AUC')
axes[1, 0].set_title('N Estimators vs PR-AUC (colored by bootstrap)')
cbar3 = plt.colorbar(scatter3, ax=axes[1, 0])
cbar3.set_label('Bootstrap')
cbar3.set_ticks([0.25, 0.75])
cbar3.set_ticklabels(['False', 'True'])

# Scatter 4: max_features vs pr_auc
scatter4 = axes[1, 1].scatter(results_df['max_features'], results_df['pr_auc'],
                               c=results_df['pr_auc'], cmap='RdYlGn', s=80, alpha=0.6)
axes[1, 1].set_xlabel('Max Features')
axes[1, 1].set_ylabel('PR-AUC')
axes[1, 1].set_title('Max Features vs PR-AUC')
plt.colorbar(scatter4, ax=axes[1, 1], label='PR-AUC')

fig.tight_layout()
plt.savefig(os.path.join(PHASE2_OUT, 'metrics', 'if_grid_scatter_analysis.png'), dpi=300)
plt.show()